# 数据集划分与标准化

流程：
1. 读取数据（负荷_天气_时间特征.csv）
2. 从 timestamp 提取月份，按月份划分训练/验证集（1-9 月训练，10-12 月验证）
3. 仅用训练集统计量进行标准化（负荷、天气等连续变量）
4. 时间特征（sin/cos编码）和布尔特征保持原值

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler
import joblib


In [2]:
base_dir = Path('data')

# 输入文件
input_path = base_dir / '负荷_天气_时间特征.csv'

# 输出文件
train_output_path = base_dir / 'train_std.csv'
val_output_path = base_dir / 'val_std.csv'

# 读取并解析时间
df = pd.read_csv(input_path, encoding='utf-8-sig')
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
df = df.dropna(subset=['timestamp']).sort_values('timestamp').reset_index(drop=True)

In [3]:
# 按月份划分（1-9月训练，10-12月验证）
month = df['timestamp'].dt.month
train_df = df[month.between(1, 9)].copy()
val_df = df[month.between(10, 12)].copy()

对负荷数据标准化处理

In [4]:
load_target_col = 'hourly_kwh_clean'
load_scaler = StandardScaler()

train_mask = train_df[load_target_col].notna()
val_mask = val_df[load_target_col].notna()

load_scaler.fit(train_df.loc[train_mask, [load_target_col]])
train_df.loc[train_mask, load_target_col] = load_scaler.transform(
    train_df.loc[train_mask, [load_target_col]]
).ravel()
val_df.loc[val_mask, load_target_col] = load_scaler.transform(
    val_df.loc[val_mask, [load_target_col]]
).ravel()

对天气特征数值数据进行标准化，独热编码的分类数据和布尔数据保留原值

In [5]:
# 连续变量标准化（仅用训练集统计量）
weather_continuous_cols = [
    '温度(℃)', '风力(级)', '风速(km/h)',
    '气压(hPa)', '湿度(%)', '能见度(km)', '云量%', '降水量对数变换'
]
weather_continuous_cols = [c for c in weather_continuous_cols if c in train_df.columns]

weather_scaler = StandardScaler()
if weather_continuous_cols:
    weather_scaler.fit(train_df[weather_continuous_cols])
    train_df[weather_continuous_cols] = weather_scaler.transform(train_df[weather_continuous_cols])
    val_df[weather_continuous_cols] = weather_scaler.transform(val_df[weather_continuous_cols])

In [6]:
# 导出
SCALER_DIR = base_dir / 'scalers'
joblib.dump(load_scaler, SCALER_DIR / 'load_scaler.joblib')
joblib.dump(weather_scaler, SCALER_DIR / 'weather_scaler.joblib')

train_df.to_csv(train_output_path, index=False, encoding='utf-8-sig')
val_df.to_csv(val_output_path, index=False, encoding='utf-8-sig')